# [기초-실습] 통계 101×데이터 분석: (4장) 추론통계~신뢰구간

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비 — 한글 폰트 설치 및 라이브러리 불러오기

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# 문제 1. 표본조사 체험하기

**📘 문제**

- 온라인 쇼핑몰은 전체 고객 수가 너무 많아, 모든 고객을 조사하기 어렵습니다.

- 그래서 무작위로 고객 30명을 뽑아 평균 만족도를 계산하고 이를 전체 만족도의 추정값으로 사용하려 합니다.

- 이번 실습에서는 직접 표본을 뽑고, 표본 평균을 구해보며,
  **“표본마다 결과가 달라질 수 있다”**는 추론 통계의 핵심 개념을 체험해봅니다.

In [ ]:
# 모집단 생성 (전체 고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.5, scale=1.2, size=10000)
population = np.clip(population, 1, 10)  # 1점 ~ 10점 사이로 제한
df_pop = pd.DataFrame({'score': population})

# 전체 모집단 시각화
sns.histplot(df_pop['score'], bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 표본을 무작위로 여러 번 뽑아 보고, 표본 평균이 어떻게 변하는지 확인해봅시다.

- 히스토그램을 그리고, 표본 평균의 분포 형태를 관찰해봅시다.

In [ ]:
# [문제 1] Q1. 모집단에서 무작위로 30명을 뽑아 표본 평균을 구해봅시다.
# 여기에 코드를 작성해주세요.

sample = df_pop['score'].sample(n=30, random_state=2025)
sample_mean = sample.mean()

print(f"표본 평균: {sample_mean:.4f}")
print(f"모집단 평균: {df_pop['score'].mean():.4f}")

In [ ]:
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.
# 여기에 코드를 작성해주세요.
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.

sample_means = []

for i in range(500):
    sample = df_pop['score'].sample(n=30, random_state=i)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print(f"반복 횟수: {len(sample_means)}")
print(f"표본 평균들의 평균: {sample_means.mean():.4f}")
print(f"표본 평균들의 표준편차: {sample_means.std():.4f}")

In [ ]:
# [문제 1] Q3. 표본 평균들의 분포를 히스토그램으로 그려봅시다. 평균선도 함께 표시해 봅시다.
# 여기에 코드를 작성해주세요.

plt.figure(figsize=(8, 5))
sns.histplot(sample_means, bins=30, kde=True)

plt.axvline(sample_means.mean(), color='red', linestyle='--', 
            label=f"표본평균들의 평균: {sample_means.mean():.3f}")
plt.axvline(df_pop['score'].mean(), color='green', linestyle='-', 
            label=f"모집단 평균: {df_pop['score'].mean():.3f}")

plt.title("표본 평균들의 분포 (500회 반복, n=30)")
plt.xlabel("표본 평균")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 평균들은 어떤 값 주변에 많이 분포해 있나요? 이 값은 전체 모집단 평균과 얼마나 비슷한가요?

- 표본을 1번 뽑았을 때와 500번을 반복해서 뽑았을 때, 표본 평균의 분포나 신뢰성에는 어떤 차이가 있나요

- 친구가 다른 표본을 뽑았다면 같은 평균이 나왔을까요? 비슷한 결과가 나왔더라도 완전히 같지 않았다면, 그 이유는 무엇일까요?

- 표본 평균들의 분포는 어떤 모양인가요? 종 모양의 정규분포처럼 보이나요? 그렇다면 왜 그렇게 되는 걸까요?

In [ ]:
# [문제 1] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# 표본 평균: 7.4047
# 모집단 평균: 7.4806

# 반복 횟수: 500
# 표본 평균들의 평균: 7.4892
# 표본 평균들의 표준편차: 0.2283

# 표본표준편차와 히스토그램으로 확인한 결과 
# 표본평균은 모평균값 주변에 분포해있고 
# 표준편차도 크지 않은것으로 보아 비슷한 편이라는 것을 알 수 있다.

# 표본평균의 평균을 구함으로써 모집단 평균에 더 가까워졌음을 알 수 있다.
# 또한 500번 추출했을 때 분포의 평균이 모평균에 가까운 정규분포의 모양을 근사하고 있다.
# 이는 500번 표본을 뽑아 평균을 구했을 때 데이터가 모평균에 가깝게 보다 안정되어간다는 것을 의미한다.

# 매번 랜덤하게 표본이 추출되므로 완전히 같은 값이 나올 수는 없다.

# 중심극한정리에 따라 표본을 뽑는 횟수가 충분히 크다면 표본평균의 분포는 정규분포로 근사한다.
# 해당분포또한 중심을 기준으로 종모양으로 대칭인 정규분포와 유사하다.

# 문제 2. 중심극한정리

**📘 문제**

- 현실에서는 모집단의 분포가 정규분포가 아닐 수도 있습니다.

- 예를 들어, 일부 고객은 매우 높은 점수를 주고, 대부분은 낮은 점수를 주는 만족도 분포가 있을 수 있죠. (예: 지수분포)

- 이처럼 원래 분포가 비정규분포여도,
  표본을 여러 번 뽑아 평균을 계산하면, 그 평균들의 분포는 정규분포에 가까워진다는 것을
  **중심극한정리(Central Limit Theorem)**라고 합니다.

- 이번 실습에서는 다양한 크기의 표본을 뽑아 평균을 계산하고,
  그 평균들의 분포가 어떻게 변하는지를 직접 실험해 봅니다.

In [ ]:
# 지수분포를 따르는 모집단 생성
np.random.seed(2025)
population = np.random.exponential(scale=50, size=100000)  # 평균 50, 비대칭 분포

# 모집단 시각화
sns.histplot(population, bins=50, kde=True)
plt.title("고객 구매 금액 분포 (모집단: 지수분포)")
plt.xlabel("구매 금액")
plt.show()

**📌 아래를 수행해 보세요:**

- 비대칭적인 모집단(지수분포)에서 무작위로 표본을 추출해 평균을 구해봅시다.

- 표본 크기를 바꿔가며, 표본 평균들의 분포가 어떻게 변화하는지 확인해봅시다.

- 히스토그램을 그리고, 분포의 모양을 관찰해봅시다.

- 표본 크기가 커질수록 표본 평균 분포의 모양과 **퍼진 정도(분산)**가 어떻게 변하는지 관찰해봅시다.

In [ ]:
# [문제 2] Q1. 모집단에서 표본을 1000번 뽑고, 각 표본의 평균을 구해봅시다.
# 표본 크기 = 5일 때

# 여기에 코드를 작성해주세요.

def get_sample_means(population, sample_size, n_repeat=1000, seed_start=0):
    means = []
    for i in range(n_repeat):
        sample = np.random.choice(population, size=sample_size, replace=False)
        means.append(sample.mean())
    return np.array(means)

np.random.seed(2025)
means_5 = get_sample_means(population, sample_size=5, n_repeat=1000)

print(f"표본 크기 5, 표본 평균들의 평균: {means_5.mean():.4f}")
print(f"표본 크기 5, 표본 평균들의 표준편차: {means_5.std():.4f}")



In [ ]:
# [문제 2] Q2. 위 과정을 표본 크기 30, 100일 때도 반복해봅시다.
# sample_size = 30, 100

# 여기에 코드를 작성해주세요.

np.random.seed(2025)
means_30 = get_sample_means(population, sample_size=30, n_repeat=1000)

np.random.seed(2025)
means_100 = get_sample_means(population, sample_size=100, n_repeat=1000)

print(f"표본 크기 30,  평균: {means_30.mean():.4f}, 표준편차: {means_30.std():.4f}")
print(f"표본 크기 100, 평균: {means_100.mean():.4f}, 표준편차: {means_100.std():.4f}")

In [ ]:
# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

# 여기에 코드를 작성해주세요.

# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sample_sizes = [5, 30, 100]
all_means = [means_5, means_30, means_100]

for ax, size, means in zip(axes, sample_sizes, all_means):
    sns.histplot(means, bins=30, kde=True, ax=ax)
    ax.axvline(means.mean(), color='red', linestyle='--', 
               label=f"평균: {means.mean():.2f}")
    ax.axvline(population.mean(), color='green', linestyle='-', 
               label=f"모평균: {population.mean():.2f}")
    ax.set_title(f"표본 크기 n={size}")
    ax.set_xlabel("표본 평균")
    ax.legend()

plt.tight_layout()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을 때 (예: 5), 평균들의 분포는 어떤 모양인가요?

- 표본 크기가 커질수록 평균 분포의 모양은 어떤 변화를 보이나요?

- 원래 모집단은 비대칭이었는데, 왜 평균들의 분포는 정규분포처럼 바뀌었을까요?

- 이 실험을 통해 중심극한정리를 어떻게 이해하게 되었나요?

- 표본 크기에 따라 **분포의 넓이(흩어짐)**는 어떻게 달라지나요?

In [ ]:
# [문제 2]데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# 표본 크기 5, 표본 평균들의 평균: 49.5647
# 표본 크기 5, 표본 평균들의 표준편차: 22.7833

# 표본 크기 30,  평균: 49.6578, 표준편차: 9.3866
# 표본 크기 100, 평균: 50.1311, 표준편차: 5.1502

# 표본의 크기가 작을 때, (현재는 표본의 크기가 5일 때) 
# 평균의 분포는 오른쪽의 꼬리가 긴 모양입니다.

# 표본의 크기가 커질 수록 표본평균의 평균을 중심으로 
# 대칭의 종모양인 정규분포의 모양으로 근사합니다.

# 모집단(지수분포)의 분포 자체는 변하지 않지만,
# 중심극한정리에 의해 표본의 크기가 커질수록 
# "표본평균들의 분포"는 정규분포에 근사합니다.

# 모집단의 분포에 상관없이 표본의 분포는 
# 표본의 크기가 커질 수록 정규분포에 가까워짐을 알 수 있었습니다.

# 표본의 크기가 커짐에 따라 표준오차의 크기는 줄어들고 있습니다.
# 이는 분포의 넓이가 작아지고 
# 표본평균이 모평균에 가깝게 분포하고 있음을 의미합니다.

# 문제 3. 표준오차

**📘 문제**

- 앞선 실습에서 우리는 **표본 크기(n)가 커질수록 표본 평균들의 분포가 더 좁아진다**는 것을 확인했습니다.
- 이처럼 표본 평균들이 얼마나 흩어져 있는지(분포의 퍼진 정도)를 나타내는 값을 **표준오차(Standard Error, SE)**라고 부릅니다.
- 표준오차는 **표본 평균들의 표준편차**와 같은 의미이며, 이는 우리가 뽑은 표본 평균이 실제 모평균과 평균적으로 얼마나 떨어져 있을지를 나타내는 **'예상 오차의 크기'**입니다.

- 통계학적으로 이 표준오차는 **`SE = σ / √n`** (모집단 표준편차 / 표본 크기의 제곱근) 이라는 공식으로 계산할 수 있습니다.
- 이 공식은 **표본 크기(n)가 커질수록 표준오차(SE)가 작아진다**는 것을 명확히 보여줍니다.

- 이번 실습에서는 여러 크기의 표본을 뽑아, 시뮬레이션을 통해 얻은 **표본 평균들의 표준편차(실험값)**가 공식으로 계산한 **표준오차(이론값)**와 얼마나 일치하는지 직접 확인해봅니다.

In [ ]:
# 모집단 생성 (평균 100, 표준편차 15)
np.random.seed(2025)
population = np.random.normal(loc=100, scale=15, size=100000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 100, 표준편차 15)")
plt.xlabel("값")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 여러 크기의 표본(10, 30, 100, 500)을 각각 1000번 뽑고, 그 평균들을 구한 뒤, **표본 평균들의 표준편차(=실험적 표준오차)**를 계산해봅시다.

- 이 결과를 이론적인 표준오차 공식과 비교하는 표를 만들고, 시각화해봅시다.

In [ ]:
# [문제 3] Q1. 표본 크기 10, 30, 100, 500에 대해 각각 1000번 표본을 뽑고, 평균을 구해봅시다.
# 각 표본 평균 분포의 표준편차를 계산해봅시다.
# 결과를 리스트에 저장하고, 표로 정리해봅시다.

def get_sample_means(population, sample_size, n_repeat=1000, seed_start=0):
    means = []
    for i in range(n_repeat):
        sample = np.random.choice(population, size=sample_size, replace=False)
        means.append(sample.mean())
    return np.array(means)

np.random.seed(2025)
means_10 = get_sample_means(population, sample_size=10, n_repeat=1000)

np.random.seed(2025)
means_30 = get_sample_means(population, sample_size=30, n_repeat=1000)

np.random.seed(2025)
means_100 = get_sample_means(population, sample_size=100, n_repeat=1000)

np.random.seed(2025)
means_500 = get_sample_means(population, sample_size=500, n_repeat=1000)

print(f"표본 크기 10, 표본 평균들의 표준편차: {means_10.std():.4f}")
print(f"표본 크기 30, 표준편차: {means_30.std():.4f}")
print(f"표본 크기 100, 표준편차: {means_100.std():.4f}")
print(f"표본 크기 500, 표준편차: {means_500.std():.4f}")

sample_sizes = [10, 30, 100, 500]
means_dict = {10: means_10, 30: means_30, 100: means_100, 500: means_500}

empirical_se = [means_dict[n].std() for n in sample_sizes]

result_df = pd.DataFrame({
    '표본크기': sample_sizes,
    '실험적 표준오차': empirical_se
})
print(result_df)
# 여기에 코드를 작성해주세요.

In [ ]:
# [문제 3] Q2. 이론적인 표준오차와 비교해봅시다.
# [공식] 표준오차(SE) = 모집단 표준편차 / √표본크기

# 여기에 코드를 작성해주세요.

population_std = population.std()

theoretical_se = [population_std / np.sqrt(n) for n in sample_sizes]

result_df['이론적 표준오차'] = theoretical_se
result_df['차이'] = result_df['실험적 표준오차'] - result_df['이론적 표준오차']

print(result_df)

In [ ]:
# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

# 여기에 코드를 작성해주세요.

# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

plt.figure(figsize=(8, 5))

plt.plot(sample_sizes, result_df['실험적 표준오차'], marker='o', label='실험적 표준오차')
plt.plot(sample_sizes, result_df['이론적 표준오차'], marker='s', linestyle='--', label='이론적 표준오차')

plt.title("표본 크기에 따른 표준오차 변화")
plt.xlabel("표본 크기 (n)")
plt.ylabel("표준오차 (SE)")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을수록, 표본 평균의 분포는 어떤 모양인가요? 넓게 퍼져 있나요?

- 표본 크기가 커질수록, 평균 분포는 어떻게 변하나요?

- 실험값과 이론값(공식 계산값)은 얼마나 비슷한가요?

- 왜 표본 크기가 커질수록 표준오차는 작아질까요?

In [ ]:
# [문제 3] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

#  표본크기  실험적 표준오차  이론적 표준오차  차이
# 0    10  4.558660  4.753921 -0.195261
# 1    30  2.689713  2.744677 -0.054965
# 2   100  1.477130  1.503322 -0.026192
# 3   500  0.661012  0.672306 -0.011294

# 표본의 크기가 작을수록 표준오차의 값이 큽니다.
# 이는 표본평균의 분포가 넓게 퍼져있음을 의미합니다.

# 표본 크기가 커질수록 표준오차가 작아지고 
# 모평균 주변에 정규분포에 가깝게 분포합니다.

# 실험값과 이론값의 차이는 표본크기 10일 때 약 -0.2,
# 표본크기 500일 때 약 -0.01로, 
# 표본 크기가 커질수록 차이도 점점 줄어들며 두 값이 매우 근소한 차이를 보입니다.

# 표준오차는 모표준편차를 표본의 크기의 제곱근 값으로 나눈 값입니다.
# 따라서 표본의 크기가 커질수록 분모가 커지므로 표준오차의 값은 작아집니다.

# 문제 4. 신뢰구간 계산과 해석

**📘 문제**

- 표본 평균은 모집단 평균을 추정하는 좋은 점 추정(Point Estimation) 값이지만, 표본오차 때문에 정확히 일치하지는 않습니다.

- 그래서 우리는 "모집단 평균이 아마 이 범위 안에 있을 것이다"라고 **구간으로 추정(Interval Estimation)**하는 것이 더 합리적입니다. 이때 사용하는 개념이 바로 **신뢰구간(Confidence Interval)**입니다.

- 신뢰구간은 표본평균 ± 오차범위 형태로 계산되며, 이 오차범위는 신뢰수준(예: 95%, 99%)과 표본오차에 의해 결정됩니다.

- 이번 실습에서는 **모집단 표준편차(σ)를 알 때(z-분포)**와 **모를 때(t-분포)**의 신뢰구간을 각각 계산해보고, 신뢰수준에 따라 구간의 폭이 어떻게 변하는지 확인해봅니다.

In [ ]:
# 모집단 생성
np.random.seed(2025)
population = np.random.normal(loc=70, scale=10, size=10000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 70, 표준편차 10)")
plt.xlabel("점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 30명을 무작위로 뽑아 평균, 표준편차, 표준오차를 계산해보세요.

- 95% 신뢰구간을 z-분포와 t-분포를 각각 사용해서 계산해보세요.

- 신뢰수준을 바꿨을 때(90%, 99%) 신뢰구간이 어떻게 변하는지 확인해보세요.

In [ ]:
# [문제 4] Q1. 모집단에서 표본 30명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 표준오차도 함께 계산해보세요.

# 여기에 코드를 작성해주세요.

np.random.seed(2025)
sample = np.random.choice(population, size=30, replace=False)

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)  # 표본표준편차 (n-1로 나눔)
se = sample_std / np.sqrt(30)

print(f"표본 평균: {sample_mean:.4f}")
print(f"표본 표준편차: {sample_std:.4f}")
print(f"표준오차(SE): {se:.4f}")

In [ ]:
# [문제 4] Q2. 모집단의 표준편차를 알고 있다고 가정하고, z-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 여기에 코드를 작성해주세요.

se_z = population_std / np.sqrt(30)   # 모표준편차 기반 SE

ci_lower = sample_mean - 1.96 * se_z
ci_upper = sample_mean + 1.96 * se_z

print(f"z-분포 기반 95% 신뢰구간: {ci_lower:.4f} <= mu <= {ci_upper:.4f}")

In [ ]:
# [문제 4] Q3. 모집단의 표준편차를 모른다고 가정하고, 표본 표준편차와 t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 여기에 코드를 작성해주세요.

from scipy import stats

df = 30 - 1  # 자유도 (n-1)
t_critical = stats.t.ppf(0.975, df)  # 95% 신뢰구간의 t 임계값

se_t = sample_std / np.sqrt(30)  # 표본표준편차 기반 SE (Q1에서 이미 계산한 값)

ci_lower_t = sample_mean - t_critical * se_t
ci_upper_t = sample_mean + t_critical * se_t

print(f"t-분포 기반 95% 신뢰구간: {ci_lower_t:.4f} <= mu <= {ci_upper_t:.4f}")
print(f"t 임계값: {t_critical:.4f} (참고: z 임계값은 1.96)")

print(f"모집단 표준편차: {population_std:.4f}")
print(f"표본 표준편차: {sample_std:.4f}")

In [ ]:
# [문제 4] Q4. 신뢰수준을 90%, 99%로 바꿔가며 신뢰구간을 계산해보고, 그 폭을 비교해봅시다.

# 여기에 코드를 작성해주세요.

from scipy import stats

df = 30 - 1  # 자유도 (n-1)
t_critical90 = stats.t.ppf(0.95, df)  # 90% 신뢰구간의 t 임계값
t_critical99 = stats.t.ppf(0.995, df)  # 99% 신뢰구간의 t 임계값

se_t = sample_std / np.sqrt(30)  # 표본표준편차 기반 SE (Q1에서 이미 계산한 값)

ci_lower_t90 = sample_mean - t_critical90 * se_t
ci_upper_t90 = sample_mean + t_critical90 * se_t

print(f"t-분포 기반 90% 신뢰구간: {ci_lower_t90:.4f} <= mu <= {ci_upper_t90:.4f}")
print(f"t 임계값: {t_critical90:.4f}")

ci_lower_t99 = sample_mean - t_critical99 * se_t
ci_upper_t99 = sample_mean + t_critical99 * se_t

print(f"t-분포 기반 99% 신뢰구간: {ci_lower_t99:.4f} <= mu <= {ci_upper_t99:.4f}")
print(f"t 임계값: {t_critical99:.4f}")

**🧠 데이터를 어떻게 읽을까요?**

- z-분포와 t-분포를 사용한 신뢰구간은 얼마나 차이가 있나요?

- 신뢰수준이 높아질수록 신뢰구간의 폭은 어떻게 변하나요? 왜 그럴까요?

- 신뢰구간이 넓다는 건 좋은 걸까요? 나쁜 걸까요?

- 이 데이터가 실제 고객 만족도라면, 신뢰구간 정보를 마케팅 전략에 어떻게 활용할 수 있을까요?

In [ ]:
# [문제 4] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# z-분포 기반 95% 신뢰구간: 63.8266 <= mu <= 74.5858

# t-분포 기반 95% 신뢰구간: 66.2654 <= mu <= 72.1469
# t 임계값: 2.0452 (참고: z 임계값은 1.96)
# 모집단 표준편차: 15.0332
# 표본 표준편차: 7.8755

# t-분포 기반 90% 신뢰구간: 66.7631 <= mu <= 71.6493
# t 임계값: 1.6991
# t-분포 기반 99% 신뢰구간: 65.2429 <= mu <= 73.1695
# t 임계값: 2.7564

# z분포 기반의 임계값은 약 1.96이고
# t분포 기반의 임계값은 약 2.04입니다.
# 다만 이번 표본은 실제 표본보다 표준편차가 작은 표본으로 추출되어
# 임계값은 더 크지만 신뢰구간은 z분포보다 좁게 나타났습니다.

# 신뢰수준이 높아질 수록 임계값이 커집니다.
# 따라서 표본평균과 표준오차에 임계값을 곱한 값의 차이로 이루어진 신뢰구간은
# 임계값이 커질수록 구간의 길이가 길어집니다.

# 신뢰구간이 넓다는 것은 설정된 신뢰도만큼
# 모평균이 신뢰구간하에 존재하도록 설계되었다는 것으로
# 신뢰도가 높을 수록 그만큼 확신도가 높아집니다.
# 추정한 구간이 넓어지면 모평균을 추정하기에 정밀도는 떨어집니다.
# 신뢰구간이 넓다고해서 좋고 나쁨을 단정할 수 없고
# 확신도와 정밀도 중 어떤 것이 필요할지에 따라 
# 신뢰도를 설정하는 것이 바람직합니다. 

# 신뢰도를 설정해 신뢰구간의 하한값을 구하고
# 이를 통해 보수적으로 전략을 세울 수 있습니다.
# 예를 들어 90% 신뢰구간의 하한값(66.8점)을 "최소 보장 만족도"로 보고
# 이를 넘지 못할 경우 마케팅 예산을 재검토하는 식의 기준으로 삼을 수 있습니다.
# 신뢰구간의 폭을 좁혀 더 정밀한 추정을 하고 싶다면
# 문제 3에서 확인했듯 표본 크기(n)를 늘리면 표준오차가 줄어들어
# 같은 신뢰수준에서도 더 좁은 구간을 얻을 수 있습니다.
# 즉, 마케팅 조사 설계 단계에서 필요한 정밀도에 맞춰
# 적절한 표본 크기를 미리 계획하는 데 활용할 수 있습니다.


# 문제 5. 미니 프로젝트 - 고객 만족도 신뢰구간 추정

**📘 문제**

- 전체 고객 10,000명을 대상으로 만족도 조사를 하는 것은 시간과 비용이 많이 듭니다.
- 그래서 우리는 무작위로 일부 고객만 조사하여, 전체 고객의 평균 만족도를 추정하려 합니다.

- 이 프로젝트에서는 실제와 같은 상황을 가정하여, 표본을 뽑고 신뢰구간을 계산한 뒤, 이 결과를 바탕으로 마케팅 전략에 어떻게 활용할 수 있을지까지 생각해보는 실습을 진행합니다.

In [ ]:
# 모집단 생성 (고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.2, scale=1.0, size=10000)
population = np.clip(population, 1, 10)

# 모집단 시각화
sns.histplot(population, bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단을 생성하고, 거기서 표본을 40명 뽑아 평균을 계산해봅시다.

- 표본 평균과 표준편차를 바탕으로 95% 신뢰구간을 계산해봅시다.

- 히스토그램을 그리고 신뢰구간을 시각화해봅시다.

- 이 결과를 어떻게 해석하고, 마케팅 전략에 활용할 수 있을지 생각해봅시다.

In [ ]:
# [문제 5] Q1. 모집단에서 표본 40명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 여기에 코드를 작성해주세요.
np.random.seed(2025)
sample = np.random.choice(population, size=40, replace=False)

sample_mean = sample.mean()
sample_std = sample.std(ddof=1)  # 표본표준편차 (n-1로 나눔)
se = sample_std / np.sqrt(40)

print(f"모집단 평균: {population.mean():.4f}")

print(f"표본 평균: {sample_mean:.4f}")
print(f"표본 표준편차: {sample_std:.4f}")
print(f"표준오차(SE): {se:.4f}")

In [ ]:
# [문제 5] Q2. 표준오차(SE)를 구하고, t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.
# 여기에 코드를 작성해주세요.

from scipy import stats

se = sample_std / np.sqrt(40)

df = 40 - 1  # 자유도 (n-1)
t_critical = stats.t.ppf(0.975, df)  # 95% 신뢰구간의 t 임계값

ci_lower_t = sample_mean - t_critical * se
ci_upper_t = sample_mean + t_critical * se

print(f"t-분포 기반 95% 신뢰구간: {ci_lower_t:.4f} <= mu <= {ci_upper_t:.4f}")
print(f"t 임계값: {t_critical:.4f} (참고: z 임계값은 1.96)")

print(f"표준오차(SE): {se:.4f}")


In [ ]:
# [문제 5] Q3. 표본 데이터의 히스토그램을 그리고, 평균 및 신뢰구간을 함께 시각화해봅시다.
#  여기에 코드를 작성해주세요.

plt.figure(figsize=(8, 5))

sns.histplot(sample, bins=15, kde=True)

plt.axvline(sample_mean, color='red', linestyle='-', 
            label=f"표본 평균: {sample_mean:.2f}")
plt.axvline(ci_lower_t, color='green', linestyle='--', 
            label=f"95% 신뢰구간 하한: {ci_lower_t:.2f}")
plt.axvline(ci_upper_t, color='green', linestyle='--', 
            label=f"95% 신뢰구간 상한: {ci_upper_t:.2f}")

plt.title("표본 만족도 분포와 95% 신뢰구간")
plt.xlabel("만족도 점수")
plt.legend()
plt.show()

In [ ]:
# [문제 5] Q4. 신뢰구간의 결과에 따라 어떤 구체적인 마케팅 전략을 세울 수 있을까요?
# 여기에 의견을 작성해주세요.

# 40명의 표본을 추출하여 평균을 나타냈을 때,
# 평균 만족도는 7.23으로 추정할 수 있습니다.
# 95% 신뢰구간은 6.94 ~ 7.51이며,
# 만약 회사의 목표 만족도가 이 하한값(6.94)보다 높게 설정되어 있다면
# 현재 데이터만으로는 목표를 달성했다고 확신 있게 말하기 어렵습니다.
# 따라서 만족도 설문조사 등을 통해 원인을 분석하고,
# 이를 개선하는 방향으로 마케팅 전략을 재수립할 필요가 있습니다.

# 지금은 40명 표본이라 신뢰구간이 다소 넓은 편인데, 
# 더 정밀한 의사결정이 필요하다면 표본 크기를 늘려 
# 신뢰구간을 좁히는 방법도 고려할 수 있습니다. 
# 다만 조사 비용과 정밀도 사이의 균형을 고려해야 합니다.

# 전체 평균뿐 아니라 고객 세그먼트별(신규/기존, 연령대 등)로 
# 표본을 나눠 신뢰구간을 비교하면,
# 어느 집단에서 만족도가 특히 낮은지 파악해 
# 타겟 마케팅에 활용할 수 있습니다.

# 매 분기 동일한 방식으로 신뢰구간을 추적하면, 
# 신뢰구간이 점점 낮아지는 추세를 조기에 발견해 
# 마케팅 대응 시점을 앞당길 수 있습니다.

**🧠 데이터를 어떻게 읽을까요?**

- 신뢰구간은 몇 점에서 몇 점 사이인가요?

- 이 구간은 전체 모집단 평균을 포함하고 있나요?

- 이 결과를 바탕으로 고객 만족도가 충분히 높다고 말할 수 있을까요?

- 만약 신뢰구간이 너무 넓게 나왔다면, 그 이유는 무엇이고 어떻게 개선할 수 있을까요?

In [ ]:
# [문제 5] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# 40명의 표본을 추출하여 평균을 나타냈을 때,
# 평균 만족도는 7.23으로 추정할 수 있습니다.
# 95% 신뢰구간은 6.94 ~ 7.51입니다.

# 모집단 평균은 7.18로 신뢰구간에 포함됩니다. 

# 만족도 7점 이상을 '충분히 높다'는 기준으로 삼는다면, 
# 신뢰구간 하한값(6.94)이 7보다 살짝 낮아 
# 확신 있게 '충분히 높다'고 말하기는 어렵습니다. 
# 기준을 6.5점으로 낮춘다면 
# 신뢰구간 전체가 이를 넘기 때문에 
# 자신 있게 말할 수 있습니다.

# 표본 크기가 작거나 표본 내 데이터의 변동성(표준편차)이 클수록 
# 신뢰구간이 넓어집니다. 
# 이번 표본의 크기는 40으로 크지 않습니다.
# 더 정밀한 의사결정이 필요하다면 표본 크기를 늘려 신뢰구간을 좁힐 수 있지만,
# 조사 비용과 정밀도 사이의 균형을 고려해야 합니다.